In [1]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

### **Qwen2.5 Model Test**

In [2]:
!pip install requests

In [3]:
import requests
import json

def stream_interviewer_response(prompt):
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": "qwen2.5:1.5b",
        "prompt": f"You are a friendly technical interviewer. Keep your response brief (under 2 sentences). User says: {prompt}",
        "stream": True # Crucial for low latency!
    }
    
    # Send request and stream the response chunk-by-chunk
    response = requests.post(url, json=payload, stream=True)
    
    print("AI Interviewer: ", end="", flush=True)
    for line in response.iter_lines():
        if line:
            # Parse the JSON line chunk
            chunk = json.loads(line.decode('utf-8'))
            token = chunk.get("response", "")
            print(token, end="", flush=True) # Stream to terminal

# Test the function
stream_interviewer_response("Hi, I am ready for my frontend developer interview.")

AI Interviewer: Great! Welcome to the interview. Ready to discuss what you've learned and how you can apply that knowledge in a real-world scenario?

### **how voice model we make by Qwen2.5 Model Test**

In [4]:
!pip install asyncio openai

In [5]:
import asyncio
import re
from openai import AsyncOpenAI

# Since Ollama mimics the OpenAI API schema, we can use the standard async client
# pointing directly to your local Ollama port.
client = AsyncOpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"  # A placeholder key is required by the SDK but ignored by Ollama
)

# Compiled regex to accurately catch sentence boundaries (. ! ?) while avoiding 
# false positives like decimals (e.g., 2.5) or abbreviations (e.g., Mr., Dr.)
SENTENCE_REGEX = re.compile(r'(?<!\bM[rs])(?<!\bDr)(?<!\bvs)(?<!\d)\.(?!\d)|[!?]')

async def stream_interviewer_response(conversation_history):
    """
    Streams tokens from local Qwen2.5, processes them through a sentence buffer,
    and yields completed sentences immediately for the TTS pipeline.
    """
    print("\n[Brain] Contacting local Qwen2.5...")
    
    response_stream = await client.chat.completions.create(
        model="qwen2.5:1.5b",
        messages=conversation_history,
        stream=True
    )
    
    token_buffer = ""
    
    async for chunk in response_stream:
        token = chunk.choices[0].delta.content
        if token:
            token_buffer += token
            
            # Check if our accumulated buffer contains a valid sentence boundary
            match = SENTENCE_REGEX.search(token_buffer)
            if match:
                # Determine where the sentence ends
                end_index = match.end()
                
                # Extract the complete sentence and clean up trailing/leading whitespace
                completed_sentence = token_buffer[:end_index].strip()
                
                # Keep the remainder of the tokens for the next round
                token_buffer = token_buffer[end_index:]
                
                # Enforce a small character threshold to prevent tiny, choppy fragments 
                # like "Oh." from cutting off natural delivery.
                if len(completed_sentence) > 5:
                    yield completed_sentence

    # Flush out any remaining tokens left in the buffer after the stream finishes
    final_fragment = token_buffer.strip()
    if final_fragment:
        yield final_fragment

# --- Simulated Test Execution ---
async def main():
    # Example state: You feed the model its identity and the history
    mock_history = [
        {
            "role": "system", 
            "content": "You are a professional tech interviewer. Keep your responses short, natural, and under two sentences. Ask the next question smoothly."
        },
        {
            "role": "user", 
            "content": "Hi, I am ready for the frontend interview. I have experience with React and Tailwind CSS."
        }
    ]
    
    print("--- Starting Local Brain Pipeline Test ---")
    
    # Simulating the exact moment the server catches complete sentences
    async for sentence in stream_interviewer_response(mock_history):
        print(f"📦 [Buffer Flush -> Send to TTS]: \"{sentence}\"")

if __name__ == "__main__":
    import sys
    
    # Check if we are running inside an active event loop (like Jupyter)
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None

    if loop and loop.is_running():
        # If in Jupyter, just schedule it directly onto the existing loop!
        print("🔄 Running inside Jupyter/Interactive environment. Scheduling task...")
        # In a notebook cell, you can also simply run: await main()
        task = loop.create_task(main())
    else:
        # Standard Python execution via command line
        asyncio.run(main())

🔄 Running inside Jupyter/Interactive environment. Scheduling task...


--- Starting Local Brain Pipeline Test ---

[Brain] Contacting local Qwen2.5...
📦 [Buffer Flush -> Send to TTS]: "Great!"
📦 [Buffer Flush -> Send to TTS]: "What specific projects or skills in using these technologies can you highlight to demonstrate your capabilities?"


### **Text to Speech Model (Kokoro)**

In [6]:
!pip install kokoro sounddevice soundfile numpy

In [7]:
from kokoro import KPipeline
import soundfile as sf

# Initialize the pipeline (using 'a' for American English)
pipeline = KPipeline(lang_code='a', repo_id='hexgrad/Kokoro-82M')

# Text returned from Ollama
text = "Welcome to your interview. Let's start with your background."

# Generate voice (using 'af_heart' or any of the 54 built-in voices)
generator = pipeline(text, voice='af_heart')

for i, (gs, ps, audio) in enumerate(generator):
    # Save the generated audio to a file
    sf.write(f"output_{i}.wav", audio, 24000)

c:\kali\product\netran ai\backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\kali\product\netran ai\backend\.venv\Lib\site-packages\torch\nn\modules\rnn.py:1011: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)
c:\kali\product\netran ai\backend\.venv\Lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


In [8]:
from IPython.display import Audio, display

# This will display an interactive audio player right in your Jupyter Notebook!
display(Audio(filename="output_0.wav", autoplay=True))

In [9]:
import time
import queue
import threading
import numpy as np
import sounddevice as sd
from kokoro import KPipeline

# 1. Initialize local TTS pipeline with explicit repo_id to silence warning
print("🎙️ Initializing local TTS pipeline...")
tts_pipeline = KPipeline(lang_code='a', repo_id='hexgrad/Kokoro-82M')
VOICE = 'af_heart'  # High-quality voice style (e.g., am_adam, af_heart)

# 2. Background worker queue to avoid locking your terminal while audio plays
audio_queue = queue.Queue()

def audio_playback_worker():
    """Background worker that continuously streams audio fragments as they finish generating."""
    while True:
        audio_data = audio_queue.get()
        if audio_data is None:  # Shutdown signal
            break
        sd.play(audio_data, samplerate=24000)
        sd.wait()  # Block background thread until segment finishes playing
        audio_queue.task_done()

# Boot up the speaker thread
playback_thread = threading.Thread(target=audio_playback_worker, daemon=True)
playback_thread.start()

# 3. Core Text Processing Wrapper
def stream_and_speak(text_generator):
    """
    Takes text pieces from an incoming engine stream, shards them into complete sentences,
    and forwards the binary segments to the audio driver asynchronously.
    """
    sentence_buffer = ""
    sentence_delimiters = (".", "?", "!", "\n")
    
    print("\n🤖 Local Voice AI: ", end="", flush=True)
    
    for token in text_generator:
        print(token, end="", flush=True)  # Print token immediately as it leaves the LLM
        sentence_buffer += token
        
        # Segment and submit audio on sentence borders or length threshold constraints
        if any(sentence_buffer.endswith(d) for d in sentence_delimiters) or len(sentence_buffer) > 115:
            clean_sentence = sentence_buffer.strip()
            if len(clean_sentence) > 1:
                generator = tts_pipeline(clean_sentence, voice=VOICE, speed=1.0)
                for _, _, audio in generator:
                    if audio is not None:
                        audio_arr = np.array(audio, dtype=np.float32)
                        audio_queue.put(audio_arr)
            sentence_buffer = ""

    # Clear out remaining context trailing behind the text cycle
    if sentence_buffer.strip():
        generator = tts_pipeline(sentence_buffer.strip(), voice=VOICE, speed=1.0)
        for _, _, audio in generator:
            if audio is not None:
                audio_arr = np.array(audio, dtype=np.float32)
                audio_queue.put(audio_arr)

# 4. Live Simulation Pipeline Target
def simulated_llm_stream():
    """Simulates raw stream intervals coming straight from a local context model."""
    chunks = [
        "Welcome ", "back! ", "Your ", "local ", "audio ", "pipeline ", 
        "is ", "now ", "fully ", "operational. ", "We ", "have ", "successfully ", 
        "silenced ", "all ", "warnings, ", "and ", "the ", "response ", "latency ", 
        "is ", "completely ", "gone."
    ]
    for chunk in chunks:
        yield chunk
        time.sleep(0.12)  # Match processing step rhythm to human conversational tracking

# Execute the local stack!
stream_and_speak(simulated_llm_stream())

🎙️ Initializing local TTS pipeline...

🤖 Local Voice AI: Welcome back! Your local audio pipeline is now fully operational. We have successfully silenced all warnings, and the response latency is completely gone.

### **Speech to Text (Wishper Faster-Whisper)**

In [10]:
!pip install faster-whisper

In [11]:
!pip install nvidia-cublas-cu12 nvidia-cudnn-cu12

^C


In [ ]:
from faster_whisper import WhisperModel

# Switch device to "cpu" and compute_type to "int8"
model_size = "tiny.en" 
stt_model = WhisperModel(model_size, device="cpu", compute_type="int8")

# Transcribe an audio file
segments, info = stt_model.transcribe("output_0.wav", beam_size=5)

for segment in segments:
    print(f"User said: {segment.text}")

User said:  Welcome to your interview. Let's start with your background.


### **FUll voice to voice (one time)**

In [ ]:
import os
import queue
import threading
import time
import numpy as np
import sounddevice as sd
import soundfile as sf
from faster_whisper import WhisperModel
from kokoro import KPipeline

# --- 1. INITIALIZE MODELS LOCAL ONLY ---
print("📥 Loading Faster-Whisper (STT)...")
stt_model = WhisperModel("tiny.en", device="cpu", compute_type="int8")

print("📥 Loading Kokoro-82M (TTS)...")
tts_pipeline = KPipeline(lang_code='a', repo_id='hexgrad/Kokoro-82M')
VOICE = 'am_adam'

# Audio queue and player setup for TTS
audio_queue = queue.Queue()

def audio_playback_worker():
    """Plays generated TTS sentences in the background sequential order."""
    while True:
        audio_data = audio_queue.get()
        if audio_data is None:
            break
        sd.play(audio_data, samplerate=24000)
        sd.wait()
        audio_queue.task_done()

# Start background audio thread
threading.Thread(target=audio_playback_worker, daemon=True).start()


# --- 2. MICROPHONE RECORDER HOOK ---
def record_microphone(duration=5, samplerate=16000, filename="user_input.wav"):
    """Records your microphone and saves it as a temporary WAV file."""
    print(f"\n🎤 Recording for {duration} seconds... Speak now!")
    # Capture audio at 16kHz (optimal for Whisper models)
    recording = sd.rec(int(duration * samplerate), samplerate=samplerate, channels=1, dtype='float32')
    sd.wait()  # Wait until recording is done
    print("🛑 Recording stopped. Transcribing...")
    
    # Save the file
    sf.write(filename, recording, samplerate)
    return filename


# --- 3. THE SPEAK ENGINE ---
def speak_text(text):
    """Feeds text immediately into Kokoro sentence by sentence."""
    generator = tts_pipeline(text, voice=VOICE, speed=1.0)
    for _, _, audio in generator:
        if audio is not None:
            audio_arr = np.array(audio, dtype=np.float32)
            audio_queue.put(audio_arr)


# --- 4. EXECUTE FULL VOICE-TO-VOICE RUN ---
def run_interaction_loop():
    # 1. Record your physical microphone
    audio_file = record_microphone(duration=4)  # Change duration as needed!
    
    # 2. Transcribe voice to text using Faster-Whisper
    segments, _ = stt_model.transcribe(audio_file, beam_size=5)
    user_text = "".join([segment.text for segment in segments]).strip()
    
    if not user_text:
        print("🤷 Could not capture any speech. Try talking closer to the mic!")
        return

    print(f"🗣️ You said: {user_text}")
    
    # 3. Simulated Response (or you can connect your local Ollama endpoint here!)
    ai_response = f"I heard you say: {user_text}. Your local AI pipeline is completely connected."
    print(f"🤖 AI Response: {ai_response}")
    
    # 4. Speak back to you
    speak_text(ai_response)

# Run a test session!
run_interaction_loop()

📥 Loading Faster-Whisper (STT)...
📥 Loading Kokoro-82M (TTS)...

🎤 Recording for 4 seconds... Speak now!
🛑 Recording stopped. Transcribing...
🗣️ You said: How are you?
🤖 AI Response: I heard you say: How are you?. Your local AI pipeline is completely connected.


### **Loop**

In [12]:
!pip install ollama

In [ ]:
import os
import queue
import threading
import time
import numpy as np
import sounddevice as sd
import soundfile as sf
import ollama
from faster_whisper import WhisperModel
from kokoro import KPipeline

# --- 1. CONFIGURATION & MODEL INITIALIZATION ---
print("📥 Loading Faster-Whisper (STT)...")
# Using the optimized CPU configuration you verified earlier
stt_model = WhisperModel("tiny.en", device="cpu", compute_type="int8")

print("📥 Loading Kokoro-82M (TTS)...")
# Suppressing the repo warning by passing repo_id explicitly
tts_pipeline = KPipeline(lang_code='a', repo_id='hexgrad/Kokoro-82M')
VOICE = 'am_adam'  # Feel free to change to 'af_heart' or any other built-in voice

# Select your local Ollama model (ensure you have run `ollama pull qwen2.5:1.5b` first)
OLLAMA_MODEL = "qwen2.5:1.5b" 

# --- 2. ASYNCHRONOUS TTS PLAYBACK ENGINE ---
audio_queue = queue.Queue()

def audio_playback_worker():
    """Plays generated TTS audio sequentially from a background thread to prevent UI freezing."""
    while True:
        audio_data = audio_queue.get()
        if audio_data is None:
            break
        sd.play(audio_data, samplerate=24000)
        sd.wait()
        audio_queue.task_done()

# Start the background audio playback manager
threading.Thread(target=audio_playback_worker, daemon=True).start()

# --- 3. HELPER FUNCTIONS ---

def record_microphone(duration=4, samplerate=16000, filename="user_input.wav"):
    """Records raw audio from your mic for a set duration and saves it as a WAV file."""
    print(f"\n🎤 Listening ({duration}s)... Speak now!")
    recording = sd.rec(int(duration * samplerate), samplerate=samplerate, channels=1, dtype='float32')
    sd.wait()  # Block main thread until the recording duration is complete
    print("🛑 Processing speech...")
    sf.write(filename, recording, samplerate)
    return filename

def speak_text(text):
    """Chunks incoming LLM text into sentences and plays them sequentially."""
    generator = tts_pipeline(text, voice=VOICE, speed=1.0)
    for _, _, audio in generator:
        if audio is not None:
            audio_arr = np.array(audio, dtype=np.float32)
            audio_queue.put(audio_arr)

# --- 4. THE LIVE CONVERSATIONAL LOOP ---

def run_voice_assistant():
    print(f"\n🚀 Eva is fully loaded! (Using {OLLAMA_MODEL})")
    print("👉 Say 'stop eva' at any point to terminate the session.\n")
    
    # Initialize conversation memory to give Eva context of previous turns
    conversation_history = [
        {
            "role": "system", 
            "content": "You are Eva, a helpful, friendly local AI voice assistant. Keep your answers brief (1-2 sentences maximum) so they can be spoken quickly and naturally."
        }
    ]
    
    while True:
        try:
            # 1. Capture user voice
            audio_file = record_microphone(duration=4)
            
            # 2. Transcribe voice using Faster-Whisper
            segments, _ = stt_model.transcribe(audio_file, beam_size=5)
            user_text = "".join([segment.text for segment in segments]).strip()
            
            # Skip empty entries if no sound was picked up
            if not user_text or len(user_text) < 2:
                print("🤷 No speech detected. Listening again...")
                continue

            print(f"🗣️ You: {user_text}")
            
            # 3. Check for global shutdown keyword
            if "stop eva" in user_text.lower():
                print("\n👋 Termination command recognized. Exiting...")
                speak_text("Goodbye! Shutting down local assistant engine now.")
                # Give the background audio worker a moment to play the exit message
                time.sleep(3.0) 
                break
                
            # 4. Append the user message to history
            conversation_history.append({"role": "user", "content": user_text})
            
            # 5. Send payload to your local Ollama instance
            print("🧠 Thinking...")
            response = ollama.chat(model=OLLAMA_MODEL, messages=conversation_history)
            ai_response = response['message']['content'].strip()
            
            # Append AI's response to history
            conversation_history.append({"role": "assistant", "content": ai_response})
            
            print(f"🤖 Eva: {ai_response}")
            
            # 6. Speak the response out loud
            speak_text(ai_response)
            
            # Tiny cool-down buffer to let audio start running before starting the next listen cycle
            time.sleep(1.0)
            
        except Exception as e:
            print(f"⚠️ Error encountered during execution loop: {e}")
            break

# --- 5. EXECUTION ---
if __name__ == "__main__":
    pass
    # Ensure you have your Ollama application running locally before executing
    # run_voice_assistant()

📥 Loading Faster-Whisper (STT)...
📥 Loading Kokoro-82M (TTS)...


c:\kali\product\netran ai\backend\.venv\Lib\site-packages\torch\nn\modules\rnn.py:1011: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)
c:\kali\product\netran ai\backend\.venv\Lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)



🚀 Eva is fully loaded! (Using qwen2.5:1.5b)
👉 Say 'stop eva' at any point to terminate the session.


🎤 Listening (4s)... Speak now!
🛑 Processing speech...
🗣️ You: Hello? Where's that?
🧠 Thinking...
🤖 Eva: I'm right here in the room by you!

🎤 Listening (4s)... Speak now!
🛑 Processing speech...
🗣️ You: That's the way to engage the...
🧠 Thinking...
🤖 Eva: Yes, of course! When someone says "That's the way to engage the..." I'd suggest they are likely talking about how to effectively communicate or connect with others.

🎤 Listening (4s)... Speak now!
🛑 Processing speech...
🗣️ You: Okay.
🧠 Thinking...
🤖 Eva: Got it! Let me know if you need anything else.

🎤 Listening (4s)... Speak now!
🛑 Processing speech...
🗣️ You: Okay, start.
🧠 Thinking...
🤖 Eva: Sure, just tell me what you're looking for and I'll help guide you on the best way to proceed.

🎤 Listening (4s)... Speak now!
🛑 Processing speech...
🗣️ You: Okay.
🧠 Thinking...
🤖 Eva: Great! How can I assist you today?

🎤 Listening (4s)... Sp

KeyboardInterrupt: 